# Bookmark migration

Tracker row **#18 (Bookmark)** — legacy Strapi `bookmarks` → new `circles`
rows (`owned_type='postcard'`, `relationship='bookmark'`). First use of the
universal Circle relationship layer.

Dependency chain (per tracker): **Postcard (#16)** and **User (#12)** —
both per-env map files must exist (`legacy_postcard_id_map`,
`legacy_user_id_map`).

Scope decisions (2026-08-10):
- `user` → `user_id`, `postcard` → `owned_id`, both via the per-env maps.
  Bookmarks whose **user or postcard is not in its map** (deleted users,
  Designer Tours postcards) are **skipped** → manual review lists.
- `createdAt` → `added_at` (when the member saved it — the one timestamp
  carried over).
- Orphan bookmarks (no user or no postcard relation in legacy) are skipped
  → printed list.
- Legacy **duplicate (user, postcard) pairs collapse** into one row via the
  Circle unique key — the earliest createdAt wins (id-sorted, DO NOTHING).
- **Dropped:** `updatedAt` (meaningless for a bookmark). Nothing else —
  legacy bookmarks carry no other fields.
- `sequence_date` / `source_enquiry_id` stay NULL — those belong to booked
  journeys, not bookmarks.
- No id map file written — nothing downstream references bookmark ids.

Run cells top to bottom. Idempotent — `ON CONFLICT DO NOTHING` on the
Circle unique key. Safe to re-run.

In [1]:
import os, json
from pathlib import Path

import requests
import psycopg
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

CMS_BASE_URL = os.environ["CMS_BASE_URL"].rstrip("/")
HEADERS = {"Authorization": f"Bearer {os.environ['CMS_API_TOKEN']}"}
DATABASE_URL = os.environ["DATABASE_URL"]
ENV_SUFFIX = {"development": "_dev", "production": "_prod"}.get(DATABASE_URL.rsplit("/", 1)[-1], "")


def attrs(item):
    """Entry fields — Strapi v4 nests them under 'attributes', v5 is flat."""
    return item.get("attributes", item)


def rel(obj):
    """Unwrap a populated relation — v4: {'data': {'attributes': {...}}}, v5: flat dict."""
    if isinstance(obj, dict) and "data" in obj:
        obj = obj["data"]
    if not obj:
        return None
    return obj.get("attributes", obj)


def fetch_all(path, params=None):
    """Fetch every page of a Strapi collection endpoint (data/meta envelope)."""
    items, page = [], 1
    while True:
        p = {"pagination[page]": page, "pagination[pageSize]": 100, "sort": "id", **(params or {})}
        r = requests.get(f"{CMS_BASE_URL}{path}", headers=HEADERS, params=p, timeout=120)
        r.raise_for_status()
        body = r.json()
        items.extend(body["data"])
        pg = body.get("meta", {}).get("pagination", {})
        if page >= pg.get("pageCount", 1):
            return items
        page += 1


conn = psycopg.connect(DATABASE_URL)
print("connected to:", DATABASE_URL.rsplit("/", 1)[-1])

# per-environment maps from the user and postcard migrations
user_map = {int(k): int(v) for k, v in
            json.loads((ROOT / f"legacy_user_id_map{ENV_SUFFIX}.json").read_text()).items()}
postcard_map = {int(k): int(v) for k, v in
                json.loads((ROOT / f"legacy_postcard_id_map{ENV_SUFFIX}.json").read_text()).items()}
print(f"loaded {len(user_map)} user mappings, {len(postcard_map)} postcard mappings ({ENV_SUFFIX or 'no suffix'})")

connected to: production
loaded 3365 user mappings, 6195 postcard mappings (_prod)


## 1. Fetch all bookmarks

`populate=*` brings `user` and `postcard` (only their ids are needed).

In [2]:
bookmarks = sorted(fetch_all("/api/bookmarks", {"populate": "*"}), key=lambda x: x["id"])
print(f"fetched {len(bookmarks)} bookmarks")

# quick shape check on the first entry
if bookmarks:
    a = attrs(bookmarks[0])
    print({k: type(v).__name__ for k, v in a.items()})

fetched 1489 bookmarks
{'id': 'int', 'createdAt': 'str', 'updatedAt': 'str', 'user': 'dict', 'postcard': 'dict'}


## 2. bookmark → `circles` (bookmark)

One row per legacy bookmark: `(user_id, 'postcard', owned_id, 'bookmark',
added_at=createdAt)`. Id-sorted + `DO NOTHING`, so for legacy duplicates the
earliest bookmark's timestamp wins.

In [3]:
conn.rollback()  # clear any aborted transaction from a previous failed run

inserted = 0
orphans, unmapped_users, unmapped_postcards = [], [], []

with conn.cursor() as cur:
    for bm in bookmarks:
        a = attrs(bm)
        u, p = rel(a.get("user")), rel(a.get("postcard"))
        if not u or not p:
            orphans.append((bm["id"], "no user" if not u else "no postcard"))
            continue

        new_uid = user_map.get(u["id"])
        if not new_uid:  # user not migrated (deleted / skipped)
            unmapped_users.append((bm["id"], u["id"], u.get("username")))
            continue

        new_pid = postcard_map.get(p["id"])
        if not new_pid:  # postcard skipped in #16 (Designer Tours)
            unmapped_postcards.append((bm["id"], p["id"], p.get("name")))
            continue

        cur.execute(
            """
            INSERT INTO circles (user_id, owned_type, owned_id, relationship, added_at)
            VALUES (%s, 'postcard', %s, 'bookmark', COALESCE(%s::timestamptz, now()))
            ON CONFLICT (user_id, owned_type, owned_id, relationship) DO NOTHING
            """,
            (new_uid, new_pid, a.get("createdAt")),
        )
        inserted += cur.rowcount

conn.commit()
print(f"bookmark circles inserted this run: {inserted}")
print(f"(duplicates collapsed by the unique key: {len(bookmarks) - inserted - len(orphans) - len(unmapped_users) - len(unmapped_postcards)})")
print(f"skipped orphans ({len(orphans)}): {orphans[:20]}")
print(f"MANUAL REVIEW legacy users not in map ({len(unmapped_users)}): {unmapped_users[:20]}")
print(f"MANUAL REVIEW legacy postcards not in map ({len(unmapped_postcards)}): {unmapped_postcards[:20]}")

bookmark circles inserted this run: 1105
(duplicates collapsed by the unique key: 2)
skipped orphans (130): [(351, 'no postcard'), (558, 'no postcard'), (561, 'no postcard'), (582, 'no postcard'), (601, 'no postcard'), (953, 'no postcard'), (1325, 'no postcard'), (1326, 'no postcard'), (1329, 'no postcard'), (1531, 'no postcard'), (1537, 'no postcard'), (1549, 'no postcard'), (1580, 'no postcard'), (1583, 'no postcard'), (1585, 'no postcard'), (1679, 'no postcard'), (1682, 'no postcard'), (1708, 'no postcard'), (1709, 'no postcard'), (1765, 'no postcard')]
MANUAL REVIEW legacy users not in map (0): []
MANUAL REVIEW legacy postcards not in map (252): [(330, 802, 'Life Lessons with a Sicilian Mamma'), (331, 803, 'Creating Firey Wines in the Shadow of Mount Etna'), (332, 807, 'The Sacred Salt Pans of Trapani'), (333, 529, 'All Aboard Mount Etna’s Historic Wine Train'), (334, 801, 'Vibrant, Chaotic Life in Palermo’s Street Markets'), (335, 808, 'A Way To Transform Spoiled Cheese into a Uni

## 3. Verification

Expected: circle count ≈ fetched bookmarks minus skips minus collapsed
duplicates; every row resolves to a real user and postcard (0 broken refs —
Circle has no DB-level FK on owned_id, so this is the app-level check).

In [ ]:
with conn.cursor() as cur:
    for label, q in [
        ("circles total",            "SELECT COUNT(*) FROM circles"),
        ("postcard bookmarks",       "SELECT COUNT(*) FROM circles WHERE owned_type = 'postcard' AND relationship = 'bookmark'"),
        ("distinct users w/ bkmks",  "SELECT COUNT(DISTINCT user_id) FROM circles WHERE owned_type = 'postcard' AND relationship = 'bookmark'"),
        ("distinct postcards bkmkd", "SELECT COUNT(DISTINCT owned_id) FROM circles WHERE owned_type = 'postcard' AND relationship = 'bookmark'"),
        ("broken postcard refs (want 0)", "SELECT COUNT(*) FROM circles c WHERE c.owned_type = 'postcard' AND c.relationship = 'bookmark' AND NOT EXISTS (SELECT 1 FROM postcards p WHERE p.id = c.owned_id)"),
        ("added_at carried (not today)",  "SELECT COUNT(*) FROM circles WHERE owned_type = 'postcard' AND relationship = 'bookmark' AND added_at < now() - interval '1 day'"),
    ]:
        cur.execute(q)
        print(f"{label:30}: {cur.fetchone()[0]}")

    # most-bookmarked postcards (top 10)
    cur.execute("""
        SELECT p.name, COUNT(*) AS n
        FROM circles c JOIN postcards p ON p.id = c.owned_id
        WHERE c.owned_type = 'postcard' AND c.relationship = 'bookmark'
        GROUP BY p.id, p.name ORDER BY n DESC, p.name LIMIT 10
    """)
    print("\nmost-bookmarked postcards:")
    for name, n in cur.fetchall():
        print(f"  {name:40}: {n}")
conn.close()